**2D**
1) Define the domain $\Omega$
2) Generate triangulations T_h
3) Construct finite element space V_h
4) Get local stiffness matrix A_k
5) Get local mass matrix M_k
6) Assemble local stiffness matrices into global stiffness matrix A
7) Assemble local mass matrices into a global mass matrix M
8) Impose Dirichlet Boundary Condition
9) Solve $A * u = \lambda * M * u$
10) Extract eigenvalues
11) compute the eigenvectors
12) Compare with exact solution


**2D FEM Implementation notes**
1. Generated mesh using Delaunay triangulation
2. Computed local element matrices
3. Assembled global stiffness and mass matrices
4. Identified boundaru and interior nodes
5. Imposed homogeneous Dirichlet boundary conditions by removing rows and columns associated with boundary nodes
6. Solved reduced generalised eigenvalue problem

*For future*
- global matrices are sparse (majority of elements are 0) and symmetric
- boundary conditions reduce the dimention of the system
- enumerate shows which node number corresponds to the coordinates

In [ ]:
# 2D let the domain be uniform and defined on [0, 1]x[0, 1]
import numpy as np
from scipy.linalg import eigh
from scipy.spatial import Delaunay
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-poster")

In [ ]:
# method for putting values on the main diagonal
def put_value_in_special_index(matrix, grad_phi_i, grad_phi_j, index):
       # find the dot product of 2 gradients
       grad_phi_i_j = np.dot(grad_phi_i, grad_phi_j)
       
       matrix.put(index, grad_phi_i_j)

       return matrix

# stiffness matrix
def stiffness_matrix_A(grad_phi):
       # create a matrix (interior_nodes x interior_nodes) of zeros
       A_lower_tri = np.zeros((3, 3), dtype=float)
       # off diagonal values
       # first make lower triangular matrix
       # A_2_1
       put_value_in_special_index(A_lower_tri, grad_phi[1], grad_phi[0], 3)
       # A_3_1
       put_value_in_special_index(A_lower_tri, grad_phi[2], grad_phi[0], 6)
       # A_3_2
       put_value_in_special_index(A_lower_tri, grad_phi[2], grad_phi[1], 7)

       A = create_symmetric_matrix(A_lower_tri)
       # diagonal values
       put_value_in_special_index(A, grad_phi[0], grad_phi[0], 0)
       put_value_in_special_index(A, grad_phi[1], grad_phi[1], 4)
       put_value_in_special_index(A, grad_phi[2], grad_phi[2], 8)
       return A

def create_symmetric_matrix(lower_tri_matrix):
       # transpose the lower triangular matrix
       lower_tri_matrix_T = lower_tri_matrix.T
       # create symmetric matrix by adding the lower triangular matrix to its transpose
       sym_matrix = lower_tri_matrix + lower_tri_matrix_T

       return sym_matrix

# mass matrix
def M_loc():
       
       return np.array(
              [
                     [2, 1, 1],
                     [1, 2, 1],
                     [1, 1, 2]
              ]
       )


# find the stiffness and mass matrices for individual triangle
def triangle_solver(coords_of_triangle):

       # find the area of the triangle
       col = np.array([1, 1, 1])
       # create 3x3 matrix to find the area
       coords_matrix = np.hstack((coords_of_triangle, np.atleast_2d(col).T))

       # area of a triangle
       T_k = 0.5 * abs(np.linalg.det(coords_matrix))

       x = []
       y = []
       # for coordinate in all of the coordinates of the nodes of this triangle
       for coord in coords_of_triangle:
              x.append(float(coord[0]))
              y.append(float(coord[1]))

       c = []   
       c.append(x[2] - x[1])
       c.append(x[0] - x[2])
       c.append(x[1] - x[0])

       b = []
       b.append(y[1] - y[2])
       b.append(y[2] - y[0])
       b.append(y[0] - y[1])

       # find the gradients of the basis functions
       grad_phi = np.array([
              (1 / (2 * T_k)) * np.array([b_i, c_i]) for b_i, c_i in zip(b, c)
       ])


       # get local stiffness matrix
       A_local = T_k * stiffness_matrix_A(grad_phi)
       
       # get local stiffness matrix
       M_local = (T_k / 12) * M_loc()
       
       return A_local, M_local

# put the triangle in the global matrix
def put_local_to_global(global_matrix, local_matrix, coord):

       n_local = local_matrix.shape[0]

       for a in range(n_local):
              for b in range(n_local):
                     global_matrix[coord[a], coord[b]] += local_matrix[a, b]
       

       return global_matrix

# get boundary and interior nodes
def get_boundary_and_interior_nodes(domain):
       boundary_nodes = []
       interior_nodes = []

       for i, (x, y) in enumerate(domain):
              # check if any of the coordinates are on the boundary
              if x == 0 or x == 1 or y == 0 or y == 1:
                     boundary_nodes.append(i)
              else:
                     interior_nodes.append(i)

       return boundary_nodes, interior_nodes

# get global matrices
def get_global_matrices(coords_of_tri, domain):
       
       n_nodes = np.max(coords_of_tri) + 1

       A_global = np.zeros((n_nodes, n_nodes), dtype=float)
       M_global = np.zeros((n_nodes, n_nodes), dtype=float)

       # for every trinagle in the mesh
       for triangle in tri_coord_sort:
              coords = domain[triangle]

              A_local, M_local = triangle_solver(coords)
              global_coords = triangle.tolist()

              put_local_to_global(A_global, A_local, global_coords)
              put_local_to_global(M_global, M_local, global_coords)

       return A_global, M_global

In [ ]:
# More complex mesh 
#######
##### MAIN #####
########
# create mesh
# define the axis intervals
x = np.linspace(0, 1, 3)
y = np.linspace(0, 1, 3)

# coordinate generation via meshgrid (takes 1D arrays and duplicates them to build 2D grids)
X, Y = np.meshgrid(x, y)

# c_ matches the first X with the first Y, second X with second Y etc
domain = np.c_[X.ravel(), Y.ravel()]
# ravel() takes 2D matrix structure and reads it row by row into a long single list of coordinates
# Delaunay cannot read 2D grid, thats why we flatten X and Y, so they can be paired together

# create triangles on the domain
tri = Delaunay(domain)
tri_coord_sort = np.sort(tri.simplices)
print(tri_coord_sort)

# visualise the triangulations
plt.triplot(domain[:,0], domain[:,1], tri.simplices.copy())
plt.plot(domain[:,0], domain[:,1], "o")

# to see the nodes on the graph
# enumerate shows which node number corresponds to the coordinates
for i, (x, y) in enumerate(domain):
       plt.text(x, y, f"P{i}", fontsize=12)

# labeling the triangles
for k, triangle in enumerate(tri.simplices):

    centroid = domain[triangle].mean(axis=0)

    plt.text(centroid[0], centroid[1], f"T{k}", color="red")

# gca - get current axes
# set_aspect("equal") prevents stretching the plot, if it's square it will look like square
plt.gca().set_aspect("equal")
plt.show()

# get global matrices
A_global, M_global = get_global_matrices(tri_coord_sort, domain)

# sanity check
print(np.allclose(A_global, A_global.T))
print(np.allclose(M_global, M_global.T))

print(A_global.sum(axis=1))


In [ ]:
# get boundary and interior nodes
boundary_nodes, interior_nodes = get_boundary_and_interior_nodes(domain)

# reducing matrices based on boundary condition, that u = 0 on the boundary
A_reduced = A_global[np.ix_(interior_nodes, interior_nodes)]
M_reduced = M_global[np.ix_(interior_nodes, interior_nodes)]

# finding eigenvalues
eigvals, eigvecs = eigh(A_reduced, M_reduced)
print(eigvals)